## Data personalization (Spring 2025)

This notebook executes one case of data transformation to multiple datasets (one source, multiple targets)

It uses the sampling/copula method prototyped in [this notebook](2024-11-25-data-personalization-prototyping.ipynb)

## LLM designator

In [1]:
# the following flag selects for LLM specific functions
llm_choices = [ "GEMINI_1.5", "GPT_4" ]
LLM = "GPT_4" #"GEMINI_1.5"
if LLM not in llm_choices:
    raise Exception(f"{LLM} is unknown")

## API keys

In [44]:
def read_key(filename):
    with open(filename, "r") as file:
        return file.read()

# put your key in text file on one line with no return
openai_api_key= read_key("openai.key")

gemini_api_key=  read_key("gemini.key")

## Functions/libraries

In [41]:
import pandas as pd
import phitter
import numpy as np

# ============================================================================
# LOGGING
# ============================================================================

# simple logging 
log = []
def add_to_log( item ):
    log.append(item)

import io

def print_to_string(*args, **kwargs):
    output = io.StringIO()
    print(*args, file=output, **kwargs)
    contents = output.getvalue()
    output.close()
    return contents
# ============================================================================
# DESCRIPTORS
# ============================================================================

# an attempt to get reasonable parameters from gpt-derived min/max and distribution
def best_distribution(var_type, dist_name,data):
    dist_name = dist_name.lower().replace(" ","_")
    name = dist_name

    # REPAIR BROKEN NAMES
    # if we have categorical type but an unknown distribution (typically b/c LLM made error), force uniform
    if var_type == "categorical" and name not in phitter.discrete.DISCRETE_DISTRIBUTIONS.keys():
        name = "uniform"
    # if we have numeric type but an unknown distribution of any type, force normal
    elif var_type == "numeric" and name not in [*phitter.continuous.CONTINUOUS_DISTRIBUTIONS.keys(), *phitter.discrete.DISCRETE_DISTRIBUTIONS.keys()]:
        name = "normal"
    # llm seems to misspecify bernoulli as binomial
    elif var_type == "numeric" and dist_name == "binomial" and data[0]==0 and data[2]==1:
        name = "bernoulli"

    if name is not dist_name:
        add_to_log(f"!!! Best distribution: repaired name '{dist_name}' to new name '{name}'")

    # in general, we only try to fit distribution proposed by LLM or one of our repair cases; TODO: consider adding additional repair cases above to this
    distributions_to_fit=[name]

    # we get a few errors from llm mispecification of continuous distributions
    # "normal" by itself is causing phitter to fail a lot
    # phitter may find a solution where it is impossible to generate samples within the target range
    # both issues can be resolved by adding some addition options, i.e. trying related distributions
    cont_dist_list = ["normal",'beta','exponential','gamma','lognormal', 'weibull']
    if name in cont_dist_list:
        distributions_to_fit = cont_dist_list

    # negative binomial is often better than poisson, so make sure it is an option
    if name == "poisson":
        distributions_to_fit.append("negative_binomial")

    # sometimes uniform is misspecified as binomial; so always add uniform as an option
    if var_type == "categorical" and name == "binomial":
        distributions_to_fit.append("uniform")

    # for unclear reasons, phitter seems to fail on binomial. Adding hypergeometric and negative_binomial as backup options
    if var_type == "numeric" and name == "binomial":
        # TODO: hypergeometric seems to not fit well with 3 datapoints
        distributions_to_fit.extend(["hypergeometric","poisson","negative_binomial"])

    # # when llm proposes uniform for numeric, it's usually done something that phitter can't resolve
    # if var_type == "numeric" and name == "uniform":
    #     distributions_to_fit.extend(["geometric","hypergeometric","negative_binomial"])

    # FINAL SANITY CHECK
    # If LLM proposed discrete distribution for continuous data, fix it now
    if name in phitter.discrete.DISCRETE_DISTRIBUTIONS.keys() and any( not isinstance(element, int) for element in data):
        add_to_log(f"!!! Discrete distribution {name} proposed for continuous data {data}: forcing to continuous")
        name = "normal"
        distributions_to_fit = ["normal",'beta','exponential','gamma','lognormal', 'weibull']

    # log what we are about to try
    add_to_log(f"Finding best distribution, LLM identified {dist_name}, repaired to {name} and trying against {distributions_to_fit}")

    # bernoulli seems to have problems fitting with 3 data points; we kluge here and expand to 6
    if name == "bernoulli": # or name == "binomial":
        data = data * 2
        
    # discrete case
    if name in phitter.discrete.DISCRETE_DISTRIBUTIONS.keys():
        phi = phitter.PHITTER(
            data=data,
            fit_type="discrete",
            # confidence_level = 0,
            distributions_to_fit=distributions_to_fit
        )
    # continuous case
    else:
        phi = phitter.PHITTER(
            data=data,
            # confidence_level = 0,
            distributions_to_fit=distributions_to_fit
        )
    phi.fit()

    # error checking; see TODO below
    if len(phi.not_rejected_distributions) == 0:
        add_to_log(f"!!! Phitter does not like any distribution for {name} with {data}")
        add_to_log(f"phi summarize: {phi.summarize()}")
        add_to_log(f"sorted sse distributions: {phi.sorted_distributions_sse}")
        if name in  name in phitter.discrete.DISCRETE_DISTRIBUTIONS.keys():
            add_to_log(f"discrete distribution instances: {phi.phitter_discrete.distribution_instances}")
        else:
            add_to_log(f"continuous distribution instances: {phi.phitter_continuous.distribution_instances}")
        
    add_to_log(f"Best fit on {data} is {phi.best_distribution}")
    # TODO there appears to be potential for error here. If we specify distributions to search and the ?SSE? is bad, phi will have nothing for us and throw an error
    # We could 1) try to check phi summarize or similar; if there are no entries run with more distributions 2) add more distributions to start
    return phi.best_distribution


# Use information in the target_descriptors to fit distributions using phitter
def add_distribution_to_descriptors( source_descriptors, target_descriptors):

    var_list = list(zip(source_descriptors, target_descriptors))
    
    descriptors_with_dist_params = []
    for org,llm in var_list:
        dist_params = None
        data = []
        # if the type is categorical, use the numeric min/max from the original description
        if llm['var_type'] == "categorical":
            llm['minimum_value'] = org['minimum_value']
            llm['maximum_value'] = org['maximum_value']
            data = [ llm['minimum_value'], llm['maximum_value'] ]
        else:
            data = [ llm['minimum_value'], llm['median_value'], llm['maximum_value'] ]
    
        dist_params = best_distribution(llm['var_type'], llm['distribution'], data)
    
        r = dict(llm, **{'dist_params':dist_params})
        descriptors_with_dist_params.append(r)

    add_to_log("==Descriptors with distribution parameters")
    add_to_log(descriptors_with_dist_params)
    return descriptors_with_dist_params

# Determine if a column should be considered categorical
def is_cat(column):
    categorical_dtypes = ['object', 'category', 'bool']
    if column.dtype.name in categorical_dtypes:
        return True
    else:
        return False   

# def convertStringColumnsToNum(data):
#     for col in data.columns:
#         if(is_cat(data[col])): 
#             tmp = pd.Categorical(data[col])
#             data[col] = tmp.codes

# Return descriptor dictionary for each column AND create a copy of the dataframe where the categorical columns have been transformed to integers
def descriptors_for_columns(a_df):
    descriptors = []
    tmp_df = pd.DataFrame()
    for col in a_df.columns:
        # print(f'processing {col}')
        # if categorical, convert and handle as discrete
        b_dist = None
        min = None
        median = None
        most_frequent = None
        max = None
        var_type = None
        if(is_cat(a_df[col])): 
            # categorical name min/max
            # tmp = pd.Categorical(a_df[col]).as_ordered()
            tmp = pd.Categorical(a_df[col]).as_ordered().codes
            min = int(tmp.min())
            counts = np.bincount(tmp)
            most_frequent = int(np.argmax(counts))
            max = int(tmp.max())
            var_type = "categorical"
            tmp_df[col] = tmp
            # b_dist = best_distribution(tmp.codes,False)
            descriptors.append( 
            { "name" : col,
              "var_type" : var_type,
              "minimum_value" : min,
              "most_frequent" : most_frequent,
              "maximum_value" : max,
              "distribution" : "UNKNOWN" #b_dist
            })
        # else handle as continuous
        else:
            # b_dist = best_distribution(a_df[col],True)
            min = a_df[col].min()
            median = a_df[col].median()
            max = a_df[col].max()
            var_type = "numeric"
            tmp_df[col] = a_df[col]
            descriptors.append( 
                { "name" : col,
                  "var_type" : var_type,
                  "minimum_value" : min,
                  "median_value" : median,
                  "maximum_value" : max,
                  "distribution" : "UNKNOWN" #b_dist
                })
    add_to_log("==Descriptors")
    add_to_log(descriptors)
    return tmp_df,descriptors
# ============================================================================
# PROMPTS
# ============================================================================
import json
import numpy as np

# for dumping descriptors into prompts; we sometimes need to convert np types
class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)

# for changing numbers to words in prompts
import inflect
p = inflect.engine()

# ================GEMINI============================
# Given dataframe descriptors and topic, create a prompt. Gemini version
def get_prompt_gemini( a_descriptors, topic ): 
    n_var = len(a_descriptors)
    list = json.dumps(a_descriptors,  cls=NpEncoder,sort_keys=True)
    # Prompt v4; remove distribution info from input; give instructions to replace UNKNOWN with correct distribution
    return f"""I have a dataset with {n_var} variables. The variables are defined in this list according to their name, minimum value, maximum value, and statistical distribution: 
    {list}
    Create a set of variables for a dataset on the topic of {topic} based on the dataset above. The {topic} dataset should have the same number of variables as the above dataset, and the variables should have the same types as the above dataset. However, the new dataset should be on the topic of {topic} with variable names appropriate for {topic}. Replace UNKNOWN with plausible statistical distributions of the new variables. Give your results in json format."""

# get possible levels for a categorical variable, gemini version
def get_categorical_prompt_gemini( variable_name, number, topic ): 
    return f"""What are {p.number_to_words(int(number))} possible {variable_name} for {topic}? Give your results in a json list."""

# # map llm generated levels of a categorical variable onto source levels, alphabetically
# def get_alphabetically_ordered_levels_prompt_gemini( df_col, candidate_list):
#     sorted_indices = np.argsort(df_col.unique())
#     candidate_list.sort()
#     return [candidate_list[i] for i in sorted_indices]

# # map llm generated levels of a categorical variable onto source levels
# def get_default_ordered_levels_prompt_gemini( df_col, candidate_list):
#     sorted_indices = np.argsort(df_col.unique())
#     candidate_list
#     return [candidate_list[i] for i in sorted_indices]

# prompt too get an ordering of levels of a categorical variable along a numeric variable for a topic
def get_categorical_order_prompt_gemini( categorical_levels, numeric_variable, topic ): 
    return f"""On the topic of {topic}, what is the expected order from lowest to highest of {categorical_levels} in terms of {numeric_variable}? Give your results in a json list."""

# ================GPT============================

# Given dataframe descriptors and topic, create a prompt.
def get_prompt_gpt( a_descriptors, topic ): 
    n_var = len(a_descriptors)
    list = json.dumps(a_descriptors,  cls=NpEncoder,sort_keys=True)
    # Prompt v4; remove distribution info from input; give instructions to replace UNKNOWN with correct distribution
    return f"""I have a dataset with {n_var} variables. The variables are defined in this list according to their name, minimum value, maximum value, and statistical distribution: 
    {list}
    Create a set of variables for a dataset on the topic of {topic} based on the dataset above. The {topic} dataset should have the same number of variables as the above dataset, and the variables should have the same types as the above dataset. However, the new dataset should be on the topic of {topic} with variable names appropriate for {topic}. Replace UNKNOWN with plausible statistical distributions of the new variables. Give your results in json format."""

# get possible levels for a categorical variable.
def get_categorical_prompt_gpt( variable_name, number, topic ): 
    return f"""What are {p.number_to_words(int(number))} possible {variable_name} for {topic}? Give your results in a json list."""

# prompt too get an ordering of levels of a categorical variable along a numeric variable for a topic
def get_categorical_order_prompt_gpt( categorical_levels, numeric_variable, topic ): 
    return f"""On the topic of {topic}, what is the expected order from lowest to highest of {categorical_levels} in terms of {numeric_variable}? Give your results in a json list."""


# ================GENERIC============================

# # Given dataframe descriptors and topic, create a prompt
# def get_prompt( a_descriptors, topic ): 
#     if LLM == "GEMINI_1.5":
#         get_prompt_gemini( a_descriptors, topic )
#     elif LLM == "GPT_4":
#         get_prompt_gpt( a_descriptors, topic )
#     else:
#         raise Exception(f"LLM {LLM} is unknown")
    
# # get possible levels for a categorical variable
# def get_categorical_prompt( variable_name, number, topic ): 
#     if LLM == "GEMINI_1.5":
#         get_categorical_prompt_gemini( variable_name, number, topic )
#     elif LLM == "GPT_4":
#         get_categorical_prompt_gpt( variable_name, number, topic )
#     else:
#         raise Exception(f"LLM {LLM} is unknown")

# def get_categorical_order_prompt( categorical_levels, numeric_variable, topic ):
#     if LLM == "GEMINI_1.5":
#         get_categorical_order_prompt_gemini( categorical_levels, numeric_variable, topic )
#     elif LLM == "GPT_4":
#         get_categorical_order_prompt_gpt( categorical_levels, numeric_variable, topic )
#     else:
#         raise Exception(f"LLM {LLM} is unknown")


# ============================================================================
# LLM API CALLS
# ============================================================================

# for failure to generate levels, back off to capital letters
import string
import time
from pydantic import BaseModel

# specify structure
class LLMVariable(BaseModel):
    distribution: str
    maximum_value: float
    median_value: float
    minimum_value: float
    name: str
    var_type: str

# we're having problems with dicts being returned without this
# also we'd like to dynamically specify the length, since it is getting ignored in the prompt,
# but that doesn't seem possible. Putting in a dummy field to cue the LLM seems plausible but produced garbage
class LLMLevels(BaseModel):
    levels: list[str]
    # num_levels: int

# flatten lists of lists to a single list
def flatten(xss):
    return [x for xs in xss for x in xs]

# check if var is a list of strings
def is_list_of_strings(var):
  return isinstance(var, list) and all(isinstance(element, str) for element in var)

#-------------------------------
# Gemini API
import os
import typing_extensions as typing
import google.generativeai as genai
from google.ai.generativelanguage_v1beta.types import content

#2024-11-25 Andrew Olney API key
genai.configure(api_key=gemini_api_key)


# Get new descriptors from gemini (internally uses these to get prompt)
def get_new_descriptors_gemini(source_descriptors,topic):
    # throttle
    time.sleep(1)
    
    # Create the model
    generation_config = {
      "temperature": 1,
      "top_p": 0.95,
      "top_k": 40,
      "max_output_tokens": 8192,
      "response_schema": list[LLMVariable],
      "response_mime_type": "application/json",
    }
    
    prompt = get_prompt_gemini( source_descriptors, topic)
    model = genai.GenerativeModel("gemini-1.5-flash-8b")
    result = model.generate_content(
        prompt,
        generation_config=generation_config
        )
    add_to_log("==Descriptors LLM call")
    add_to_log(prompt)
    add_to_log("==Descriptors LLM response")
    add_to_log(result.text)
    return json.loads(result.text)

def get_categorical_levels_gemini( variable_name, number, topic ):
    # throttle
    time.sleep(1)
    
    generation_config = {
      "temperature": 1,
      "top_p": 0.95,
      "top_k": 40,
      "max_output_tokens": 8192,
      "response_schema": LLMLevels,
      "response_mime_type": "application/json",
    }
    prompt = get_categorical_prompt_gemini( variable_name, number, topic )
    model = genai.GenerativeModel("gemini-1.5-flash-8b")
    result = model.generate_content(
        prompt,
        generation_config=generation_config
        )
    add_to_log("==Categorical levels LLM call")
    add_to_log(prompt)
    add_to_log("==Categorical levels LLM response")
    add_to_log(result.text)
    return json.loads(result.text)

def get_categorical_levels_ordered_against_numeric_col_gemini(categorical_col_name,number,numeric_col_name,topic):
    # throttle
    time.sleep(1)
    
    generation_config = {
        "temperature": 1,
        "top_p": 0.95,
        "top_k": 40,
        "max_output_tokens": 8192,
        "response_schema": LLMLevels,
        "response_mime_type": "application/json",
    }
    # get possible labels for integer labels of categorical variable for given topic
    levels = get_categorical_levels_gemini(categorical_col_name,number,topic)
    levels = levels['levels']
    print(f"debug/{number} levels of {categorical_col_name}: {levels}")
    
    # handle generation error - back off to letters of the alphabet if we do not have n distinct levels
    if len(set(levels)) != number:
        add_to_log(f"!!! Generated {levels} but need {number} distinct levels, setting levels to uppercase alphabet characters")
        print(f"debug/Generated {levels} but need {number} distinct levels, setting levels to uppercase alphabet characters")
        levels = list(string.ascii_uppercase[:int(number)])

    # order these variables from low to high on numeric variable for a given topic
    prompt = get_categorical_order_prompt_gemini(levels,numeric_col_name,topic)
    model = genai.GenerativeModel("gemini-1.5-flash-8b")
    result = model.generate_content(
        prompt,
        generation_config=generation_config
        )
    add_to_log("==Ordered categorical levels LLM call")
    add_to_log(prompt)
    add_to_log("==Ordered categorical levels LLM response")
    add_to_log(result.text)
    ordered_levels = json.loads(result.text)['levels']
    # handle generation error
    if len(ordered_levels) != number:
        add_to_log(f"!!! Generated ordered {ordered_levels} but need {number} ordered levels, setting ordered levels to levels")
        print(f"debug/Generated ordered {ordered_levels} but need {number} ordered levels, setting ordered levels to levels")
        ordered_levels = levels
    return ordered_levels

#-------------------------------
# GPT

from openai import OpenAI
import json

# Schema is only valid on newer models
# text={
#     "format": {
#         "type": "json_schema",
#         "name": "llm_variable",
#         "schema": {
#             "type": "object",
#             "properties": {
#                 "distribution": {
#                     "type": "string"
#                 },
#                 "maximum_value": {
#                     "type": "float"
#                 },
#                 "median_value": {
#                     "type": "float"
#                 },
#                 "minimum_value": {
#                     "type": "float"
#                 },
#                 "name": {
#                     "type": "string"
#                 },
#                 "var_type": {
#                     "type": "string"
#                 }
#             },
#             "required": ["distribution","maximum_value","median_value","minimum_value","name", "var_type"],
#             "additionalProperties": False
#         },
#         "strict": True
#     }
# },
    
# Get new descriptors (internally uses these to get prompt)
def get_new_descriptors_gpt(source_descriptors,topic):
    # throttle
    time.sleep(1)

    prompt = get_prompt_gpt( source_descriptors, topic)
    
    client = OpenAI(api_key=openai_api_key)
    response = client.responses.create(
        model="gpt-4-turbo-2024-04-09",
        input=prompt,
        text={
            "format": {
              "type": "json_object"
            }
          },
        reasoning={},
        tools=[],
        temperature=0,
        max_output_tokens=4096,
        top_p=0.95,
        store=False
    )

    add_to_log("==Descriptors LLM call")
    add_to_log(prompt)
    add_to_log("==Descriptors LLM response")
    add_to_log(response.output_text)

    # check json
    raw_result = json.loads(response.output_text)
    clean_result = None
    # if we got a dict, then we assume the values are the variables we want
    if type(raw_result) is dict:
        clean_result = raw_result.values()
        clean_result = flatten(clean_result)
    else:
        clean_result = flatten(raw_result)
    return clean_result

def get_categorical_levels_gpt( variable_name, number, topic ):
    # throttle
    time.sleep(1)

    prompt = get_categorical_prompt_gpt( variable_name, number, topic )
    
    client = OpenAI(api_key=openai_api_key)
    response = client.responses.create(
        model="gpt-4-turbo-2024-04-09",
        input=prompt,
        text={
            "format": {
              "type": "json_object"
            }
          },
        reasoning={},
        tools=[],
        temperature=0,
        max_output_tokens=4096,
        top_p=0.95,
        store=False
    )

    add_to_log("==Categorical levels LLM call")
    add_to_log(prompt)
    add_to_log("==Categorical levels LLM response")
    add_to_log(response.output_text)

    # check json
    raw_result = json.loads(response.output_text)
    clean_result = None
    # if we got a dict, then we assume the values are the variables we want
    if type(raw_result) is dict:
        clean_result = raw_result.values()
        clean_result = flatten(clean_result)
    else:
        clean_result = flatten(raw_result)

    # double check that we do not have a list of dicts; if we do, take the first attribute from each dict and make a list of that
    if all(isinstance(element, dict) for element in clean_result):
        # TODO: one could make an LLM call here to repair the situation, e.g. "Extract TeamLeague from the following JSON {JSON} Return a JSON list"
        clean_result = [ list(d.items())[0][1] for d in clean_result]
    
    return clean_result

def get_categorical_levels_ordered_against_numeric_col_gpt(categorical_col_name,number,numeric_col_name,topic):
    # throttle
    time.sleep(1)

    # get possible labels for integer labels of categorical variable for given topic
    levels = get_categorical_levels_gpt(categorical_col_name,number,topic)
    print(f"debug/{number} levels of {categorical_col_name}: {levels}")
    
    # handle generation error - back off to letters of the alphabet if we do not have n distinct levels        
    if len(set(levels)) != number:
        add_to_log(f"!!! Generated {levels} but need {number} distinct levels, setting levels to uppercase alphabet characters")
        print(f"debug/Generated {levels} but need {number} distinct levels, setting levels to uppercase alphabet characters")
        levels = list(string.ascii_uppercase[:int(number)])
        
    # order these variables from low to high on numeric variable for a given topic
    prompt = get_categorical_order_prompt_gpt(levels,numeric_col_name,topic)
    
    client = OpenAI(api_key=openai_api_key)
    response = client.responses.create(
        model="gpt-4-turbo-2024-04-09",
        input=prompt,
        text={
            "format": {
              "type": "json_object"
            }
          },
        reasoning={},
        tools=[],
        temperature=0,
        max_output_tokens=4096,
        top_p=0.95,
        store=False
    )
    
    add_to_log("==Ordered categorical levels LLM call")
    add_to_log(prompt)
    add_to_log("==Ordered categorical levels LLM response")
    add_to_log(response.output_text)

    # check json
    raw_result = json.loads(response.output_text)
    clean_result = None
    # if we got a dict, then we assume the values are the variables we want
    if type(raw_result) is dict:
        clean_result = raw_result.values()
        clean_result = flatten(clean_result)
    else:
        clean_result = flatten(raw_result)
 
    ordered_levels = clean_result
    
    # handle generation error
    if len(ordered_levels) != number:
        add_to_log(f"!!! Generated ordered {ordered_levels} but need {number} ordered levels, setting ordered levels to levels")
        print(f"debug/Generated ordered {ordered_levels} but need {number} ordered levels, setting ordered levels to levels")
        ordered_levels = levels
    return ordered_levels

# GENERIC

# return the first numeric descriptor
def find_first_numeric(descriptors):
    for item in descriptors:
        if item['var_type'] == 'numeric':
            return item['name']
    return None  # Return None if no match is found

# get new descriptors from llm based on old
def get_new_descriptors(source_descriptors,topic):

    # use llm to get labels for the integer categories
    if LLM == "GEMINI_1.5":
        descriptors = get_new_descriptors_gemini(source_descriptors,topic)
    elif LLM == "GPT_4":
        descriptors = get_new_descriptors_gpt(source_descriptors,topic)
    else:
        raise Exception(f"LLM {LLM} is unknown")
    return descriptors

# categoricals at this stage have integer levels. use the llm to come up with level labels that are ordered against a numeric variable, then assign these labels to the integer levels
def assign_levels_to_categoricals(descriptors, topic, df_target):
    # a numeric col to anchor on
    num_col_name = find_first_numeric(descriptors)

    if num_col_name is not None:
        for item in descriptors:
            if item['var_type'] == 'categorical':
                cat_col_name = item['name']
                # number of levels 
                num_levels = df_target[cat_col_name].max() + 1
                
                # use llm to get labels for the integer categories
                if LLM == "GEMINI_1.5":
                    level_labels = get_categorical_levels_ordered_against_numeric_col_gemini( cat_col_name, num_levels, num_col_name, topic)
                elif LLM == "GPT_4":
                    level_labels = get_categorical_levels_ordered_against_numeric_col_gpt( cat_col_name, num_levels, num_col_name, topic)
                else:
                    raise Exception(f"LLM {LLM} is unknown")

                # print(target_categories)
                print(f'debug/ordered level labels by {num_col_name}: {level_labels}')
                # order of integer codes, from low to high, on this numeric col
                sort_order = df_target_correlated[ [num_col_name, cat_col_name ] ].groupby(cat_col_name).median().reset_index().sort_values(by=num_col_name)[cat_col_name].values
                # order the llm labels according to this order
                sorted_level_labels = [x for _, x in sorted(zip(sort_order, level_labels))]
                
                # handle generation error
                if sorted_level_labels == []:
                    sorted_level_labels = level_labels
                    add_to_log(f"!!! Failure to order {level_labels} by {num_col_name}, setting sorted_level_labels = level_labels")
                    print(f"debug/Failure to order {level_labels} by {num_col_name}, setting sorted_level_labels = level_labels")
                elif len(sorted_level_labels) != len(level_labels):
                    sorted_level_labels = level_labels
                    add_to_log(f"!!! Length of sorted_level_labels not equal to level_labels, setting sorted_level_labels = level_labels")
                    print(f"debug/Length of sorted_level_labels not equal to level_labels, setting sorted_level_labels = level_labels")
                
                print(f'debug/assigned level labels by {sort_order}: {sorted_level_labels}')
                # replace numeric categorical codes with LLM labels
                df_target[cat_col_name] = pd.Categorical.from_codes(df_target[cat_col_name].astype('Int64'), sorted_level_labels)
                add_to_log("==Assign levels to categoricals")
                add_to_log(num_col_name)
    add_to_log(df_target)
    return df_target

# ============================================================================
# SAMPLING 
# ============================================================================

import scipy
# the parameter names come from phitter and match scipy positional args by position
generator_dict = {
    'bernoulli' : { 'function': scipy.stats.bernoulli, 'parameters': ['p']},
    # beta, scale = 1 / beta
    'beta' : { 'function': scipy.stats.beta, 'parameters': ['alpha','beta']},
    'binomial' : { 'function': scipy.stats.binom, 'parameters': ['n','p']},
    # expo, scale is 1/lambda
    'exponential' : { 'function': scipy.stats.expon, 'parameters': ['lambda']},
    'gamma' : { 'function': scipy.stats.gamma, 'parameters': ['a','b']},
    'geometric' : { 'function': scipy.stats.geom, 'parameters': ['p']},
    'hypergeometric' : { 'function': scipy.stats.hypergeom, 'parameters': ['n','k','p']},
    'logarithmic' : { 'function': scipy.stats.logser, 'parameters': ['p']},
    'lognormal' : { 'function': scipy.stats.lognorm, 'parameters': ['mu','sigma']},
    'logistic' : { 'function': scipy.stats.logistic, 'parameters': ['mu','sigma']},
    'normal': { 'function': scipy.stats.norm, 'parameters':  ['mu','sigma'] },
    'negative_binomial' : { 'function': scipy.stats.nbinom, 'parameters':  ['r','p'] },
    'poisson' : { 'function': scipy.stats.poisson, 'parameters': ['lambda']},
    'uniform' : { 'function': scipy.stats.randint, 'parameters': ['a','b']}, #discrete uniform!
    # 'uniform' : { 'function': scipy.stats.uniform, 'parameters': ['a','b']} #continuous uniform
    'weibull' : { 'function': scipy.stats.weibull_max, 'parameters': ['alpha','beta']}
}

# ABANDONED SCIPY B/C MAPPING SCIPY TO PHITTER WAS PITA - functional forms differed
def scipy_generate_samples(r,target_size):
    sample_size = target_size * 100  # overgenerate because we filter later
    
    dist_name = r['dist_params']['id']
    # this assumes parameters are in the right order
    params = list(r['dist_params']['parameters'].values())
    
    generator = generator_dict[dist_name]['function']
    #special case: for uniform, make upper bound inclusive
    if dist_name == 'uniform': #if dist_name in ['uniform','bernoulli']:
        params[-1] += 1
    add_to_log(f"Sampling {dist_name} with {params}")
    samples = generator.rvs(*params, size=sample_size)
    return samples


from phitter import simulation

def phitter_generate_samples(r,target_size):
    # ( distribution, params, sample_count ):
    sample_size = target_size * 10  # overgenerate because we filter later
    dist_name = r['dist_params']['id']
    params = r['dist_params']['parameters']
   
    add_to_log(f"Sampling {dist_name} with {params}")

    s = simulation.ProcessSimulation()
    s.add_process(
        prob_distribution=dist_name,
        parameters=params,
        process_id="VAR",
        new_branch=True,
    )
    s.run(number_of_simulations=sample_size)
    return s['VAR'].values.tolist()

from random import sample
from collections import defaultdict

def generate_column_from_specification(r,target_size):
    dist_name = r['dist_params']['id']
    params = list(r['dist_params']['parameters'].values())
    min_val = r['minimum_value']
    max_val = r['maximum_value']
    # samples = scipy_generate_samples(r,target_size)
    samples = phitter_generate_samples(r,target_size)

    add_to_log(f"Generated {len(samples)} raw samples for {dist_name} with min {min(samples)} and max {max(samples)}")
    
    # Ensure samples are within the specified range
    #special case: if uniform, ensure equal number of levels
    if dist_name == 'uniform':
        levels = params[-1]-params[0]
        n = target_size/levels
        d = defaultdict(int)
        temp = []
        for x in samples:
            if d[x] < n:
                temp.append(x)
            d[x] += 1
        samples = temp
    else:
        samples = [ x for x in samples if min_val <= x <= max_val ]

    # log possible sampling issues
    add_to_log(f"Target row count is {target_size}")
    add_to_log(f"Available samples is {len(samples)} using min {min_val} and max {max_val}")
    
    # downsample in order to get desired number
    samples = sample(samples,target_size)
    add_to_log("==Generated column from specification")
    add_to_log(dist_name)
    add_to_log(params)
    add_to_log("==")
    return samples

def generate_dataframe_from_specification(fitted_target_descriptors, target_row_count):
    df_target = pd.DataFrame()
    for d in fitted_target_descriptors:
        add_to_log(f"=={d['name']} column, beginning generation")
        samples = generate_column_from_specification(d,target_row_count)
        df_target.insert(len(df_target.columns),d['name'],samples)
    add_to_log("==Generate dataframe from specification")
    add_to_log(df_target)
    return df_target

# ============================================================================
# COPULA METHOD
# ============================================================================

import numpy as np
import pandas as pd
from scipy.stats import norm, rankdata

# Function to transform data to uniform distribution
def to_uniform(data):
    ranks = rankdata(data, method='average')
    # uniform_data = (ranks - 0.5) / len(data)
    uniform_data = (ranks) / len(data)
    return uniform_data

# Function to transform uniform data back to original distribution
def from_uniform(uniform_data, original_data):
    sorted_original = np.sort(original_data)
    return sorted_original[np.searchsorted(np.sort(uniform_data), uniform_data)]

def apply_gaussian_copula( df_source, df_target): 
    original_data = df_target
    
    # Transform original data to uniform distribution
    uniform_data = original_data.apply(to_uniform)
    
    # Apply desired correlation structure (example: identity matrix for simplicity)

    #drop na to prevent na in correlations
    desired_corr = np.corrcoef(df_source.dropna(), rowvar=False) 
    
    # condition the correlation matrix. 
    # cholesky requires positive definite but correlations are only positive semi definite
    # by adding to the diagonal we push near zero eigenvalues away from zero and achieve positive definite
    np.fill_diagonal(desired_corr, desired_corr.diagonal() + .00001)

    # get the cholesky and matrix multiply with the uniformed data to transfer structure
    cholesky_decomp = np.linalg.cholesky(desired_corr)
    correlated_uniform_data = uniform_data @ cholesky_decomp.T

    # Transform uniform data back to original distributions, one column at a time
    df_target_cor = pd.DataFrame()
    for i,name in enumerate(original_data.columns):
        target_col = from_uniform(correlated_uniform_data.iloc[:, i], original_data.iloc[:,i])
        df_target_cor.insert(len(df_target_cor.columns),name,target_col)
        
    add_to_log("==Copula transformed dataframe")
    add_to_log(df_target_cor)
    return df_target_cor

# Batch execution

In [42]:
datasets = [ "iris", "boston", "baseball", "cancer" ]
topics = [ 'News','Games','Health','Sports','Travel' ]

# Make datasets x topics new datasets
for dataset_name in datasets:
    print(f'= Source dataset: {dataset_name}')
    
    for topic_name in topics:
        print(f'== Target dataset: {topic_name}')
        
        log = []
        add_to_log(f"Dataset name {dataset_name}")
        add_to_log(f"Topic name {topic_name}")
        
        # Load dataset
        df = pd.read_csv(f"datasets/{dataset_name}.csv")

        # Get descriptor dictionary and version of dataframe where categorical columns have been converted to integers
        df_np,source_descriptors = descriptors_for_columns(df)
        
        # Prompt LLM for new descriptors
        target_descriptors = get_new_descriptors(source_descriptors,topic_name)
        
        # Infer distributions from target descriptors
        fitted_target_descriptors = add_distribution_to_descriptors( source_descriptors, target_descriptors)
        
        # Generate uncorrelated data given inferred distributions
        df_target_uncorrelated = generate_dataframe_from_specification(fitted_target_descriptors, len(df) )
        
        # Apply correlation structure using copula method
        df_target_correlated = apply_gaussian_copula(df_np, df_target_uncorrelated)
        
        # Create level labels for categorical levels that are appropriately ordered on an anchor numeric variable
        final_target_df = assign_levels_to_categoricals(target_descriptors, topic_name, df_target_correlated)
        
        # Write target df to file
        import time
        file_name_prefix = time.strftime("%Y-%m-%d_%H-%M-%S") + f"_llm_{LLM}_source_{dataset_name}_target_{topic_name}"
        final_target_df.to_csv(file_name_prefix + ".csv",index=False)
        
        # Write log to file, matching name
        with open(file_name_prefix + ".log", "w") as file:
            lines = [ print_to_string(e) for e in log ]
            file.writelines( lines )

        print(f'... done')

= Source dataset: cancer
== Target dataset: Sports
... done
== Target dataset: Travel
... done


# Stepwise execution

## Load dataset

In [4]:
log = []
datasets = [ "iris", "boston", "baseball", "cancer" ]
dataset_name = datasets[0] # "iris"

df = pd.read_csv(f"datasets/{dataset_name}.csv")

add_to_log(f"Dataset name {dataset_name}")

## Identify topic

In [5]:
topics = [ 'News','Games','Health','Sports','Travel' ]
topic = topics[0] #"Sexually transmitted infections" #"travel to Japan" #"baseball"

add_to_log(f"Topic name {topic}")

## Get descriptor dictionary and version of dataframe where categorical columns have been converted to integers

In [6]:
df_np,source_descriptors = descriptors_for_columns(df)

In [7]:
# for debug
# descriptors
# df_np

## Prompt LLM for new descriptors

In [8]:
# for debug
# get_prompt( source_descriptors, topic)

In [22]:
target_descriptors = get_new_descriptors(source_descriptors,topic)

In [24]:
# for debug
# get_categorical_prompt( "Position", 3, "baseball")
# get_categorical_levels( "Team", 3, "baseball")
# target_descriptors#['variables']
# log

## Infer distributions from target descriptors

In [25]:
# Use information in the target_descriptors to fit distributions using phitter
fitted_target_descriptors = add_distribution_to_descriptors( source_descriptors, target_descriptors)

## Generate uncorrelated data given inferred distributions

In [26]:
df_target_uncorrelated = generate_dataframe_from_specification(fitted_target_descriptors, len(df) )

In [27]:
# for debug
# df_target_uncorrelated
# df_target_uncorrelated.describe()
# df_target_uncorrelated.corr()
# import plotly.express as px
# px.histogram(df_target_uncorrelated.HomeRuns)
# log

## Apply correlation structure using copula method

In [28]:
df_target_correlated = apply_gaussian_copula(df_np, df_target_uncorrelated)

In [106]:
# for debug
# df_target_correlated
# df_target_correlated.describe()
# df_target_correlated.corr()

## Create level labels for categorical levels that are appropriately ordered on an anchor numeric variable

In [62]:
# for debug
# sort_order = df_target_correlated[ ['BattingAverage','Team' ] ].groupby('Team').median().reset_index().sort_values(by='BattingAverage',ascending=False)['Team'].values
# llm_levels = ['Atlanta Braves', 'New York Yankees', 'Los Angeles Dodgers']
# print(sort_order)
# [x for _, x in sorted(zip(sort_order, llm_levels))]

In [31]:
final_target_df = assign_levels_to_categoricals(target_descriptors, topic, df_target_correlated)

debug/3.0 levels of NewsCategory: ['Politics', 'Sports', 'Technology']
debug/ordered level labels by WordCount: ['Sports', 'Technology', 'Politics']
debug/assigned level labels by [0. 1. 2.]: ['Sports', 'Technology', 'Politics']


In [32]:
# write target df to file
import time
file_name_prefix = time.strftime("%Y-%m-%d_%H-%M-%S") + f"_llm_{LLM}_source_{dataset_name}_target_{topic}"
final_target_df.to_csv(file_name_prefix + ".csv",index=False)
# write log to file, matching name
with open(file_name_prefix + ".log", "w") as file:
    lines = [ print_to_string(e) for e in log ]
    file.writelines( lines )